# Imports

In [46]:
import mfl as mfl
import pandas as pd
import numpy as np
import mfl.api.data_loaders as mfldata
import nfl_data_py as nfl

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score, precision_recall_curve, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

from xgboost import XGBClassifier, XGBRFClassifier

from catboost import CatBoostClassifier, CatBoostRegressor

import keras
from keras.layers import Dense, ReLU, Bidirectional, Normalization, Dropout, Input
from keras.models import Sequential

import torch 
import torch.nn 

from sklearn.cluster import KMeans

import requests
from bs4 import BeautifulSoup
import time

import seaborn as sns
import matplotlib.pyplot as plt

import math
import pickle
import joblib
import os

# Cleaning

In [583]:
def scrape_NFL_REF_QB(player_name):

    first_name = player_name.split(' ')[0].lower()
    last_name = player_name.split(' ')[1].lower()
    player_url = f'https://www.sports-reference.com/cfb/players/{first_name}-{last_name}-1.html'
    if player_name == "Josh Allen":
        player_url = f'https://www.sports-reference.com/cfb/players/{first_name}-{last_name}-7.html'
    html_content = requests.get(player_url).text

    if len(player_name.split(' ')) > 2:
        first_name = player_name.split(' ')[0].lower()
        last_name = player_name.split(' ')[1].lower()
        suffix = player_name.split(' ')[2].lower()
        player_url = f'https://www.sports-reference.com/cfb/players/{first_name}-{last_name}-{suffix}-1.html'
        html_content = requests.get(player_url).text


    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find('table', {'id': 'passing_standard'})

    if not table:
        for idx in range(2, 6):
            time.sleep(3)
            first_name = player_name.split(' ')[0].lower()
            last_name = player_name.split(' ')[1].lower()
            player_url = f'https://www.sports-reference.com/cfb/players/{first_name}-{last_name}-{idx}.html'
            html_content = requests.get(player_url).text
            
            soup = BeautifulSoup(html_content, 'html.parser')
            table = soup.find('table', {'id': 'passing_standard'})
            
            if not table:
                print(f"Passing stats table not found for {player_name}. Failed on index {idx}")
            else:
                print(f"Player found on index {idx}")
                break
    
    try:
        table.find_all('th')
        headers = [th.getText() for th in table.find_all('th')]
        yrs = [i for i in headers if ('2' in i) & (len(i) == 4 or len(i) == 5)]
        career_idx = headers.index(yrs[-1]) + 1
        baseline_headers = []
        headers = [th.getText() for th in table.find_all('th')]
        rows = []

        for tr in table.find_all('tr')[1:]:
            cells = [td.getText() for td in tr.find_all('td')]
            if cells: 
                rows.append(cells)

        career_stats = []

        for row in rows:
            if row.count('') == 2:
                career_stats.extend(row)
                break

        career_stats.remove('')
        career_stats.remove('')

        career_stats.append(len(yrs))
        career_stats.append(player_name)

        column_names = [
        'G',        
        'Cmp',      
        'Att',      
        'Cmp%',     
        'Yds',      
        'TD',       
        'TD%',      
        'Int',      
        'Int%',     
        'Y/A',      
        'AY/A',     
        'Y/C',      
        'Y/G',      
        'Rate',
        'seasons',
        'name'
        ]

        final = pd.DataFrame({name: [value] for name, value in zip(column_names, career_stats)})

        if len(career_stats) == len(column_names):
            final = pd.DataFrame({name: [value] for name, value in zip(column_names, career_stats)})
        else:
            print("Error: Number of stats does not match number of column names.")

        return final    
    except:
        print(f"FAILED: On {player_name}")

In [11]:
years = range(2010, 2023)
data = nfl.import_weekly_data(years)
draft = nfl.import_draft_picks(years)

Downcasting floats.


In [286]:
all_qbs = draft[draft['position'] == 'QB']
all_qb_names = all_qbs['pfr_player_name'].unique().tolist()

In [ ]:
raw_data = []
for name in all_qb_names:
    print(name)
    raw_data.append(scrape_NFL_REF_QB(player_name=name))
    time.sleep(3)

In [11]:
extracted_data = pd.concat([i for i in raw_data if i is not None])

In [16]:
processing = data[data['player_display_name'].isin(all_qb_names)].sort_values(['player_display_name', 'season']).groupby(['player_display_name', 'recent_team'], sort=False).agg({'season':'nunique'}).reset_index()
final_years_1st_team = processing.groupby('player_display_name').agg({'season':'first'})

In [17]:
draft_teams = data[data['player_display_name'].isin(all_qb_names)].groupby(['player_display_name']).agg({'recent_team':'first'})

In [18]:
df_with_teams = pd.merge(left=extracted_data, 
                         right=draft_teams, 
                         left_on='name',
                         right_on='player_display_name',
                         how='left')

In [19]:
df_with_draft_teams = pd.merge(left=all_qbs[['pfr_player_name', 'round', 'pick', 'season', 'allpro', 'seasons_started']],
                               right=df_with_teams,
                               left_on='pfr_player_name',
                               right_on='name',
                               how='left'
                               )

In [20]:
final_years_1st_team = final_years_1st_team.rename(columns={'season':'seasons_with_draft_team'})

In [21]:
final = pd.merge(left=df_with_draft_teams,
                 right=final_years_1st_team,
                 left_on='name',
                 right_on='player_display_name',
                 how='left')

In [22]:
final = final[['pfr_player_name', 'round', 'pick', 'season', 'G', 'Cmp', 'Att', 'Cmp%',
       'Yds', 'TD', 'TD%', 'Int', 'Int%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate',
       'seasons','recent_team', 'seasons_with_draft_team']]

In [23]:
final = final.rename(columns={'season':'draft_year','pfr_player_name':'player_name', 'seasons':'college_seasons'})

In [122]:
final = pd.read_csv("~/Desktop/mfl_project/mfl/data/full_qb_dataset_v2.csv")

In [ ]:
pd.merge(left=final, right=draft.drop(['round', 'pick']), left_on='player_name', right_on='pfr_player_name', how='left')

In [ ]:
final[final['player_name'] == "Cam Newton"].columns

In [27]:
final.to_csv("~/Desktop/mfl_project/mfl/data/full_qb_dataset_v2.csv",index=False)

In [145]:
final = pd.read_csv("~/Desktop/mfl_project/mfl/data/full_qb_dataset_v2.csv")

# Modeling Pipeline

In [146]:
def map_years_with_draft_team(x):
        if x >= 4:
            return 1
        else: 
            return 0
        
def map_seasons_started(x):
        if x >= 3:
            return 1
        else: 
            return 0
    
def preprocess(df):
    df = df[df['draft_year'] <= 2019].dropna()
    numeric_features = ['G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%', 'Int', 'Int%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'college_seasons', 'pick']
    categorical_features = ['recent_team']
    ordinal_features = ['round']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('ord', OrdinalEncoder(), ordinal_features)
        ],
        remainder='drop' 
    )

    final_transformed = pd.DataFrame(preprocessor.fit_transform(df))
    final_transformed['seasons_with_draft_team'] = df['seasons_with_draft_team'].values

    return pd.DataFrame(final_transformed), preprocessor, numeric_features + categorical_features + ordinal_features
    
def score(y_test, y_probs, y_preds):
    accuracy = accuracy_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)
    roc_auc = roc_auc_score(y_test, y_probs)
    recall = recall_score(y_test, y_preds)
    precision = precision_score(y_test, y_preds)

    metric_dict = {
        'accuracy' : accuracy,
        'f1' : f1,
        'roc_auc': roc_auc,
        'recall': recall,
        'precision': precision
    }

    return metric_dict


def catboost(df, year_cutoff=2019, feature_set=None, kfold=False, folds=2):

    df = df.dropna()
    df = df[df['draft_year'] <= year_cutoff]

    if feature_set is None:
        X = df.drop(['player_name', 'pfr_player_name', 'seasons_with_draft_team', 'seasons_started'],axis=1)
        y = df['seasons_started']
    elif feature_set is not None:
        pass
    
    X = X
    y = y
    y_mapped = y.apply(map_years_with_draft_team)
    y_mapped = y.apply(map_seasons_started)

    X_train, X_test, y_train, y_test = train_test_split(X, y_mapped, test_size=.25, stratify=y_mapped, shuffle=True)

    model = CatBoostClassifier(one_hot_max_size=15,
                                iterations=500, 
                                cat_features=X.select_dtypes(include='object').columns.tolist())
    
    model.fit(X_train, y_train)

    y_preds = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:,1]

    metrics = score(y_test, y_probs, y_preds)
    
    model.fit(X, y_mapped)
    fit_model = model
    model_results = pd.DataFrame(metrics, index=[0])

    return model, model_results
    

def predict_2025_qb(model, player_name, round, pick, recent_team, predictors, season=2025):
    season = season
    variant_features = ['round', 'pick', 'draft_year']
    available_features = np.setdiff1d(model.feature_names_[:-1], variant_features).tolist()
    predictors = predictors[available_features]
    
    initial_features = pd.DataFrame({
        'round' : round,
        'pick' : pick,
        'draft_year' : season
    }, index=[0])
    
    processing = pd.concat([initial_features, predictors], axis=1)
    processing['recent_team'] = recent_team
    prediction = model.predict_proba(processing).tolist()[0][1]

    result_dict = {
         'player_name' : player_name,
         'round' : round,
         'pick' : pick,
         'prob' : prediction
    }
    
    return pd.DataFrame(result_dict, index=[0])

In [147]:
model_data, preprocessor, columns = preprocess(final)

In [118]:
final_with_responses = pd.merge(left=final, 
                                right=draft[['pfr_player_name', 'seasons_started']], 
                                left_on='player_name',
                                right_on='pfr_player_name',
                                how='left')

In [124]:
final['recent_team']

0       LA
1      DEN
2      CAR
3      CLE
4      PHI
      ... 
144    NaN
145    PIT
146    BUF
147    NaN
148    DEN
Name: recent_team, Length: 149, dtype: object

In [135]:
final['pick'].unique()

array([  1,  25,  48,  85, 122, 155, 168, 176, 181, 204, 209, 239, 250,
         8,  10,  12,  35,  36,  74, 135, 152, 160, 180, 208,   2,  22,
        57,  75,  88, 102, 185, 243, 253,  16,  39,  73,  98, 110, 112,
       115, 221, 234, 237, 249,   3,  32,  62, 120, 163, 164, 178, 183,
       194, 213, 214,  89, 103, 147,  26,  51,  91,  93, 100, 139, 162,
       187, 191, 201, 207, 223,  20,  86,  94, 137, 144, 241, 247, 262,
        11,  15,  64,  66,  67, 133, 218,   5,   6,  53, 125, 167, 189,
       224, 231, 240, 244,  42, 104, 166, 197,   7,  76, 108, 171, 199,
       203, 219, 220,  52,  87, 215])

In [119]:
draft_order = pd.read_excel("~/Downloads/Top Prospects for the 2025 NFL Draft-2.xlsx", sheet_name='2025 NFL Draft Order', skiprows=1)

abbreviations = ['TEN', 'CLE', 'NYG', 'NE', 'JAX', 'LV', 'NYJ', 'CAR', 'NO', 'CHI', 'SF', 'DAL', 'MIA', 'IND', 'ATL', 'ARI', 'CIN', 'SEA', 'TB', 'DEN', 'PIT', 'LAC', 'GB', 'MIN', 'HOU', 'LAR', 'BAL', 'DET', 'WAS', 'BUF', 'KC', 'PHI']
teams = draft_order['Team'].unique()

team_mapping = {i:j for i,j in zip(teams, abbreviations)}

draft_order['abbrev'] = draft_order['Team'].apply(lambda team: team_mapping[team])

# OK GOOD!

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

X = model_data.drop(columns=['seasons_with_draft_team'])
y = model_data['seasons_with_draft_team'].apply(map_years_with_draft_team)

model = XGBClassifier()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

scaler_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/scaler1.pkl'
model_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/xgb1.pkl'
model.fit(X, y)

with open(scaler_path, 'wb') as path:
    pickle.dump(preprocessor, path)

with open(model_path, 'wb') as path:
    pickle.dump(model, path)

test = final[final['player_name'] == 'Baker Mayfield'][columns]
test['recent_team'] = "CLE"
model.predict_proba(preprocessor.transform(test))

Cross-validation scores: [0.70588235 0.875      0.75       0.75       0.75      ]
Mean accuracy: 0.7661764705882353


array([[0.01441717, 0.9855828 ]], dtype=float32)

In [213]:
scaler_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/scaler1.pkl'
model_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/xgb1.pkl'
model.fit(X, y)

with open('scaler_path', 'wb') as path:
    pickle.dump(preprocessor, path)

In [ ]:
path

In [201]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

X_tensor = X
y_tensor = y.values.squeeze()

num_classes = len(y.unique())
input_dim = X_tensor.shape[1]

dataset = tf.data.Dataset.from_tensor_slices((X_tensor, y_tensor))
dataset = dataset.shuffle(buffer_size=len(X_tensor)).batch(16)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(input_dim,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

print("--- Training... ---")
model.fit(dataset, epochs=100)
print("--- Training complete ---")

model_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/nn.pkl'

with open(model_path, 'wb') as path:
    pickle.dump(model, path)

test = final[final['player_name'] == 'Baker Mayfield'][columns]
test['recent_team'] = "LAR"
model.predict(preprocessor.transform(test))

--- Training... ---
Epoch 1/100


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 752us/step - accuracy: 0.3795 - loss: 0.8233
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 595us/step - accuracy: 0.5225 - loss: 0.7036
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5180 - loss: 0.6859
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 583us/step - accuracy: 0.6476 - loss: 0.6147
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 754us/step - accuracy: 0.7308 - loss: 0.5675
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 588us/step - accuracy: 0.7907 - loss: 0.5385
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - accuracy: 0.7968 - loss: 0.5107
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 709us/step - accuracy: 0.7966 - loss: 0.5141
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step - accuracy: 0.8093 - loss: 0.5052
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 581us/step - accuracy: 0.7970 - loss: 0.4890
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step - accuracy: 0.8714 - loss: 0.4694
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 550us/step - accuracy: 0.8363 - loss

array([[0.99179095]], dtype=float32)

In [207]:
scaler_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/scaler.pkl'
model_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/xgb.pkl'
# model_path = f'/Users/benstager/Desktop/mfl_project/mfl/api/saved_models/nn.pkl'

In [208]:
with open(scaler_path, 'rb') as file_path:
    scaler = pickle.load(file_path)
with open(model_path, 'rb') as file_path:
    model = pickle.load(file_path)

test = final[final['player_name'] == 'Baker Mayfield'][columns]
test['recent_team'] = "NO"
model.predict(preprocessor.transform(test))

array([1])

In [180]:
pd.DataFrame(columns).to_csv('/Users/benstager/Desktop/mfl_project/mfl/data/cols.csv', index=False)

In [133]:
pd.read_csv('/Users/benstager/Desktop/mfl_project/mfl/data/cols.csv').values[:,0]

array(['G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%', 'Int', 'Int%',
       'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'college_seasons',
       'recent_team', 'round', 'pick'], dtype=object)

In [176]:
final['player_name'].values

array(['Sam Bradford', 'Tim Tebow', 'Jimmy Clausen', 'Colt McCoy',
       'Mike Kafka', 'John Skelton', 'Jonathan Crompton', 'Rusty Smith',
       'Dan LeFevour', 'Tony Pike', 'Levi Brown', 'Sean Canfield',
       'Zac Robinson', 'Cam Newton', 'Jake Locker', 'Blaine Gabbert',
       'Christian Ponder', 'Andy Dalton', 'Colin Kaepernick',
       'Ryan Mallett', 'Ricky Stanzi', 'T.J. Yates', 'Nathan Enderle',
       'Tyrod Taylor', 'Greg McElroy', 'Andrew Luck',
       'Robert Griffin III', 'Ryan Tannehill', 'Brandon Weeden',
       'Brock Osweiler', 'Russell Wilson', 'Nick Foles', 'Kirk Cousins',
       'Ryan Lindley', 'B.J. Coleman', 'Chandler Harnish', 'EJ Manuel',
       'Geno Smith', 'Mike Glennon', 'Matt Barkley', 'Ryan Nassib',
       'Tyler Wilson', 'Landry Jones', 'Brad Sorensen', 'Zac Dysert',
       'B.J. Daniels', 'Sean Renfree', 'Blake Bortles', 'Johnny Manziel',
       'Teddy Bridgewater', 'Derek Carr', 'Jimmy Garoppolo',
       'Logan Thomas', 'Tom Savage', 'Aaron Murray', 

In [197]:
test = final[final['player_name'] == 'Johnny Manziel'][columns]
test['recent_team'] = "LAC"
model.predict_proba(scaler.transform(test))

array([[9.9919552e-01, 8.0447906e-04]], dtype=float32)

In [123]:
final

,player_name,round,pick,draft_year,G,Cmp,Att,Cmp%,Yds,TD,...,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,college_seasons,recent_team,seasons_with_draft_team
0,Sam Bradford,1,1,2010,31.0,604.0,893.0,67.6,8403.0,88.0,...,16.0,1.8,9.4,10.57,13.9,271.1,175.6,3.0,LA,4.0
1,Tim Tebow,1,25,2010,55.0,661.0,995.0,66.4,9285.0,88.0,...,16.0,1.6,9.3,10.38,14.0,168.8,170.8,4.0,DEN,2.0
2,Jimmy Clausen,2,48,2010,35.0,695.0,1110.0,62.6,8148.0,60.0,...,27.0,2.4,7.3,7.33,11.7,232.8,137.2,3.0,CAR,1.0
3,Colt McCoy,3,85,2010,53.0,1157.0,1645.0,70.3,13253.0,112.0,...,45.0,2.7,8.1,8.19,11.5,250.1,155.0,4.0,CLE,3.0
4,Mike Kafka,4,122,2010,30.0,408.0,637.0,64.1,4265.0,19.0,...,20.0,3.1,6.7,5.88,10.5,142.2,123.9,4.0,PHI,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,C.J. Beathard,3,104,2017,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,Joshua Dobbs,4,135,2017,37.0,614.0,999.0,61.5,7138.0,53.0,...,29.0,2.9,7.1,6.90,11.6,192.9,133.2,4.0,PIT,2.0
146,Nathan Peterman,5,171,2017,36.0,398.0,663.0,60.0,5236.0,47.0,...,17.0,2.6,7.9,8.16,13.2,145.4,144.6,4.0,BUF,2.0
147,Brad Kaaya,6,215,2017,38.0,721.0,1189.0,60.6,9972.0,69.0,...,24.0,2.0,8.4,8.64,13.8,262.4,146.2,3.0,NaN,NaN


In [13]:
model, results = catboost(final_with_responses, year_cutoff=2018)

abbreviations = ['TEN', 'CLE', 'NYG', 'NE', 'JAX', 'LV', 'NYJ', 'CAR', 'NO', 'CHI', 'SF', 'DAL', 'MIA', 'IND', 'ATL', 'ARI', 'CIN', 'SEA', 'TB', 'DEN', 'PIT', 'LAC', 'GB', 'MIN', 'HOU', 'LAR', 'BAL', 'DET', 'WAS', 'BUF', 'KC', 'PHI']
teams = draft_order['Team'].unique()

team_mapping = {i:j for i,j in zip(teams, abbreviations)}

draft_order['abbrev'] = draft_order['Team'].apply(lambda team: team_mapping[team])

Learning rate set to 0.005594
0:	learn: 0.6905363	total: 58.9ms	remaining: 29.4s
1:	learn: 0.6876776	total: 59.7ms	remaining: 14.9s
2:	learn: 0.6841041	total: 60.4ms	remaining: 10s
3:	learn: 0.6815373	total: 61.1ms	remaining: 7.57s
4:	learn: 0.6789978	total: 61.7ms	remaining: 6.11s
5:	learn: 0.6760843	total: 62.3ms	remaining: 5.13s
6:	learn: 0.6726062	total: 63ms	remaining: 4.44s
7:	learn: 0.6698892	total: 64.3ms	remaining: 3.95s
8:	learn: 0.6663537	total: 64.9ms	remaining: 3.54s
9:	learn: 0.6632403	total: 65.5ms	remaining: 3.21s
10:	learn: 0.6599708	total: 66.1ms	remaining: 2.94s
11:	learn: 0.6578506	total: 66.8ms	remaining: 2.71s
12:	learn: 0.6554644	total: 67.3ms	remaining: 2.52s
13:	learn: 0.6515782	total: 68ms	remaining: 2.36s
14:	learn: 0.6486741	total: 68.6ms	remaining: 2.22s
15:	learn: 0.6458245	total: 69.3ms	remaining: 2.1s
16:	learn: 0.6423486	total: 69.9ms	remaining: 1.99s
17:	learn: 0.6399048	total: 70.7ms	remaining: 1.89s
18:	learn: 0.6365933	total: 71.2ms	remaining: 1.8s


NameError: name 'draft_order' is not defined

In [ ]:
abbreviations = ['TEN', 'CLE', 'NYG', 'NE', 'JAX', 'LV', 'NYJ', 'CAR', 'NO', 'CHI', 'SF', 'DAL', 'MIA', 'IND', 'ATL', 'ARI', 'CIN', 'SEA', 'TB', 'DEN', 'PIT', 'LAC', 'GB', 'MIN', 'HOU', 'LAR', 'BAL', 'DET', 'WAS', 'BUF', 'KC', 'PHI']
teams = draft_order['Team'].unique()

team_mapping = {i:j for i,j in zip(teams, abbreviations)}

draft_order['abbrev'] = draft_order['Team'].apply(lambda team: team_mapping[team])

In [ ]:
math.ceil(224/32)

In [ ]:
results

In [769]:
def predict_full_draft(player_name, draft_order):
    season = 2025
    player_data = mfldata.scrape_NFL_REF_QB(player_name)
    results = []
    for pick, team in zip(draft_order['Pick'], draft_order['abbrev']):
        if pick <= 224:
            round = math.ceil(pick/32)
        else:
            round = 7
        result = predict_2025_qb(model=model, 
                                 player_name=player_name, 
                                 round=round, 
                                 pick=pick,
                                 recent_team=team,
                                 predictors=player_data,
                                 season=season)
        results.append(result)
    final = pd.concat(results)

    return final

In [831]:
qbs_of_interest = [
    'Cameron Ward',
    'Shedeur Sanders',
    'Jalen Milroe',
]

In [ ]:
modeled_results = pd.concat([predict_full_draft(player_name=qb, draft_order=draft_order) for qb in qbs_of_interest])

In [834]:
ward_ewers = ['Cameron Ward', 'Shedeur Sanders', 'Jalen Milroe']

In [ ]:
modeled_results

In [ ]:
idx = modeled_results.groupby(['player_name', 'round'])['prob'].idxmax()
modeled_results.loc[idx, ['player_name', 'round', 'pick', 'prob']].reset_index(drop=True)


In [ ]:
draft_order

In [ ]:
cut_qb.iloc[3]

In [ ]:
modeled_results['player_name'].unique()

In [ ]:
import matplotlib.pyplot as plt

df = modeled_results[(modeled_results['round'] == 2) & (modeled_results['pick'] == 33)]

colors = ['green' if name == 'Jaxson Dart' else 'skyblue' for name in df['player_name']]

plt.figure(figsize=(10, 6))
plt.bar(df['player_name'], df['prob'], color=colors)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Probability')
plt.title('Round 2, Pick 33: Success Probabilities by Player')
plt.tight_layout()
plt.show()


In [ ]:
modeled_results[modeled_results['pick'] == 33]['prob']

In [ ]:
cut_qb = modeled_results[modeled_results['player_name'] == 'Jalen Milroe']

# Plot the line
plt.plot(cut_qb['pick'], cut_qb['prob'], label='Probability Curve')

# Highlight the first point
plt.scatter(cut_qb['pick'].iloc[20], cut_qb['prob'].iloc[20], color='Orange', label='Steelers', zorder=5)
plt.scatter(cut_qb['pick'].iloc[32], cut_qb['prob'].iloc[32], color='Red', label='Browns', zorder=5)
plt.scatter(cut_qb['pick'].iloc[33], cut_qb['prob'].iloc[33], color='Green', label='Giants', zorder=5)

plt.title(f"Jalen Milroe: Pick # vs. Success Probability")
plt.xlabel("Pick")
plt.ylabel("Probability")
plt.legend()
plt.show()

print(cut_qb.head(50))

In [ ]:
for qb in qbs_of_interest:
    cut_qb = modeled_results[modeled_results['player_name'] == qb]
    plt.plot(cut_qb['pick'], cut_qb['prob'])
    plt.title(f"{qb}: Pick # vs. Success Probability")
    plt.xlabel("Pick")
    plt.ylabel("Probability")
    plt.show()
    print(cut_qb.head(50))

In [679]:
final = pd.read_csv("/Users/benstager/Desktop/mfl_project/mfl/data/full_qb_dataset_v2.csv")

In [ ]:
results

In [ ]:
predict_2025_qb(model, player_name='Shedeur Sanders', recent_team='NO', round=1, pick=9, season=2025)

In [727]:
draft_order = pd.read_excel("~/Downloads/Top Prospects for the 2025 NFL Draft-2.xlsx", sheet_name='2025 NFL Draft Order', skiprows=1)

In [ ]:
draft_order['Team'].unique()

In [729]:
abbreviations = ['TEN', 'CLE', 'NYG', 'NE', 'JAX', 'LV', 'NYJ', 'CAR', 'NO', 'CHI', 'SF', 'DAL', 'MIA', 'IND', 'ATL', 'ARI', 'CIN', 'SEA', 'TB', 'DEN', 'PIT', 'LAC', 'GB', 'MIN', 'HOU', 'LAR', 'BAL', 'DET', 'WAS', 'BUF', 'KC', 'PHI']
teams = draft_order['Team'].unique()

team_mapping = {i:j for i,j in zip(teams, abbreviations)}

draft

In [732]:
draft_order['abbrev'] = draft_order['Team'].apply(lambda team: team_mapping[team])

In [ ]:
draft_order

# Let's try to predict out of sample

In [55]:
prospect_names = pd.read_excel("~/Downloads/Top Prospects for the 2025 NFL Draft.xlsx", sheet_name=0, skiprows=2)

In [242]:
draft_order = pd.read_excel("~/Downloads/Top Prospects for the 2025 NFL Draft.xlsx", sheet_name=2, skiprows=1)[['Pick', 'Team']]

In [ ]:
draft_order.head(50)

In [56]:
qbs_2025 = prospect_names[prospect_names['Position'] == "QB"]

In [ ]:
qbs_2025['Name']

# Analysis

In [288]:
final = pd.read_csv("/Users/benstager/Desktop/mfl_project/mfl/data/full_qb_dataset_v2.csv")

In [ ]:
max(final['draft_year'])

In [ ]:
final.columns

In [ ]:
draft.columns

In [644]:
final_with_responses = pd.merge(left=final, 
                                right=draft[['pfr_player_name', 'allpro', 'seasons_started', 'probowls', 'w_av']], 
                                left_on='player_name',
                                right_on='pfr_player_name',
                                how='left')

### Teams who picked qbs most

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=final, x='recent_team', order=final['recent_team'].value_counts().index)
plt.xticks(rotation=45)
plt.title("QBs Taken by Team (Since 2010)")
plt.xlabel('Team')
plt.ylabel('Amount Taken')
plt.show()

In [ ]:
sns.kdeplot(data=final[final['recent_team'] == 'CLE'], x='round')

In [ ]:
sns.kdeplot(data=final[final['recent_team'] == 'NYJ'], x='round')

### Qbs picked by round

In [ ]:
for round in final['round'].unique():
    print(round)
    cut = final[final['round'] == round]
    plt.figure(figsize=(12, 6))
    sns.countplot(data=cut, x='recent_team', order=cut['recent_team'].value_counts().index)
    plt.xticks(rotation=45)
    plt.show()

### Average seasons started by round

In [ ]:
import matplotlib.pyplot as plt

means = final_with_responses.groupby('round')['seasons_started'].mean()

plt.figure(figsize=(10, 6))
plt.plot(means.index, means.values, marker='o')

for x, y in zip(means.index, means.values):
    plt.text(x, y, f'{y:.2f}', ha='center', va='bottom')

plt.title("Average Seasons a Quarterback Starts by Round")
plt.xlabel('Round')
plt.ylabel('Average Seasons Started')
plt.show()

In [ ]:
final[final['TD%'] == 100]

In [545]:
no_JA = final_with_responses[final_with_responses['player_name'] != 'Josh Allen']

In [ ]:
no_JA.groupby

In [ ]:
final_with_responses.groupby('round')['allpro'].sum().plot()

In [ ]:
final_with_responses.groupby('round')['probowls'].sum().plot()

In [ ]:
round_colors = {
    1: 'gold',
    2: 'silver',
    3: 'brown',
    4: 'green',
    5: 'blue',
    6: 'purple',
    7: 'gray'
}

# Create scatter plot
plt.figure(figsize=(10, 6))
for r in range(1, 8):
    subset = no_JA[no_JA['round'] == r]
    plt.scatter(subset['pick'], subset['Rate'],
                label=f'Round {r}',
                color=round_colors[r])

In [ ]:
final_with_responses.columns

In [ ]:
final_with_responses['']

In [ ]:
final_with_responses['seasons_started'].hist()

In [ ]:
plt.scatter(no_JA['round'], no_JA['seasons_started'])

In [ ]:
import matplotlib.pyplot as plt

years = final.groupby('draft_year')['player_name'].count()

plt.figure(figsize=(10, 6))
plt.plot(years.index, years.values, marker='o')

for x, y in zip(years.index, years.values):
    plt.text(x, y, f'{y:}', ha='center', va='bottom')

plt.xlabel('Round')
plt.ylabel('Average Seasons Started')
plt.show()

In [ ]:
final[final['round'] == 1].groupby('draft_year')['player_name'].count().plot()

In [ ]:
import matplotlib.pyplot as plt

counts = final.groupby(['draft_year', 'round'])['player_name'].count().reset_index()

plt.figure(figsize=(12, 6))
for rnd in counts['round'].unique():
    subset = counts[counts['round'] == rnd]
    plt.plot(subset['draft_year'], subset['player_name'], marker='o', label=f'Round {rnd}')

plt.xlabel('Draft Year')
plt.ylabel('Number of Players')
plt.legend(title='Round')
plt.show()

In [ ]:
all_qbs.columns

In [ ]:
all_qbs.groupby('round')['probowls'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

means = final.groupby('round')['seasons_with_draft_team'].mean()

plt.figure(figsize=(10, 6))
plt.plot(means.index, means.values, marker='o')

max_x = means.idxmax()
max_y = means.max()

plt.scatter(max_x, max_y, color='red', s=100)

plt.xlabel('Round')
plt.ylabel('Average Seasons with Draft Team')
plt.show()

In [ ]:
sns.histplot(data=final[final.seasons_with_draft_team >=4], x='Y/G', stat='density')
sns.histplot(data=final[final.seasons_with_draft_team < 4], x='Y/G', stat='density')

In [ ]:
sns.countplot(final[final.seasons_with_draft_team >=4], x='college_seasons')

In [ ]:
sns.countplot(final[final.seasons_with_draft_team < 4], x='college_seasons')

In [444]:
positive_samples = final[final['seasons_with_draft_team'] >= 4]
negative_samples = final[final['seasons_with_draft_team'] < 4]

In [ ]:
positive_samples['round'].value_counts(normalize=True).plot()
negative_samples['round'].value_counts(normalize=True).sort_index().plot()

In [ ]:
final

In [ ]:
KMeans(n_clusters=3)

In [ ]:
KMeans(n_clusters=3).fit(X=final.select_dtypes(exclude='object').dropna())

In [ ]:
negative_samples.describe()

In [ ]:
positive_samples.describe()

In [ ]:
sns.heatmap(negative_samples.describe())

In [ ]:
final_with_responses['seasons_with_draft_team'].hist()

In [ ]:
final_with_responses['seasons_started'].hist()

In [554]:
first_rounders = final_with_responses[final_with_responses['round'] == 1]

In [ ]:
first_rounders.groupby('draft_year')['player_name'].count().plot()

In [ ]:
sns.heatmap(final_with_responses.select_dtypes(exclude='object').corr(), cmap='coolwarm')

In [568]:
pick_start = pd.DataFrame(final_with_responses.groupby('pick')['seasons_started'].mean().reset_index())

In [ ]:
sns.kdeplot(x=pick_start['pick'], y=pick_start['seasons_started'])

In [ ]:
sns.kdeplot(data=final_with_responses, x='pick')

In [ ]:
final_with_responses.select_dtypes(exclude='object')

In [ ]:
no_JA.groupby('draft_year')['Rate'].mean().plot()
plt.title("Draft Year vs. Average QB Rate")
plt.xlabel("Year")
plt.ylabel("Avg. Rate")

In [ ]:
no_JA.groupby('draft_year').agg({'Rate':'max', 'player_name':'first'}).reset_index()

In [ ]:
import matplotlib.pyplot as plt

# Plotting the line
no_JA.groupby('draft_year').agg({'Rate':'max', 'player_name':'first'}).reset_index().plot(x='draft_year', y='Rate', kind='line', legend=False)
plt.title("Draft Year vs. Max QB Rate")
plt.xlabel("Year")
plt.ylabel("Avg. Rate")

# Annotating each point with the player's name
for i, row in no_JA.groupby('draft_year').agg({'Rate':'max', 'player_name':'first'}).reset_index().iterrows():
    plt.text(row['draft_year'], row['Rate'] + 1, row['player_name'], 
             ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
no_JA.groupby('draft_year')['Rate', 'player_name'].max().plot()
plt.title("Draft Year vs. Average QB Rate")
plt.xlabel("Year")
plt.ylabel("Avg. Rate")

In [ ]:
max(no_JA.Rate)

In [ ]:
no_JA['response'] = no_JA['seasons_started'].apply(map_seasons_started)

In [620]:
max_pick = final_with_responses.groupby('draft_year')['pick'].min().reset_index()

In [634]:
responses_by_first = final_with_responses[(final_with_responses['pick'].isin(max_pick['pick'])) & (final_with_responses['draft_year'].isin(max_pick['draft_year']))]


In [ ]:
responses_by_first

In [ ]:
all_qbs['college'].value_counts()

In [ ]:
responses_by_first

In [ ]:
final_with_responses.value_counts(normalize=True).plot.bar()
plt.xticks(rotation=360)
plt.xlabel("Started 3 years?")
plt.ylabel("%")
plt.title("% of First Picked QBs since 2010 who started more than 3 years")

In [ ]:
responses_by_first

In [ ]:
final_with_responses.groupby('draft_year')['Rate'].max()

In [ ]:
for col in no_JA.select_dtypes(exclude='object').columns:
    no_JA.groupby('response')[col].mean().plot.bar()
    plt.title(col)
    plt.show()

In [ ]:
final_with_responses.value_counts().plot.bar()

In [905]:
final_with_responses = final_with_responses['seasons_started'].apply(map_seasons_started)

In [ ]:
final_with_responses['response']